**Level 5: Simulation and Optimization**

**Executive Summary**

Simulation and optimization are powerful scientific computing techniques used to evaluate system behaviour and identify the best decisions under varying conditions. In irrigation management, simulations can predict soil moisture changes over time, while optimization techniques can determine the most efficient use of limited water resources.

This notebook develops irrigation simulations using numerical methods and applies optimization techniques to improve water allocation across agricultural zones. The results help identify strategies that maximize water-use efficiency while maintaining target soil moisture levels.

**1. Introduction**

Agricultural systems operate under uncertainty due to changing weather conditions, crop requirements, and water availability.

Simulation allows us to:

* Model future system behaviour.
* Evaluate different irrigation strategies.
* Predict moisture changes over time.

Optimization helps:

* Minimize water wastage.
* Reduce operational costs.
* Improve irrigation efficiency.
* Allocate resources effectively.

**2. Learning Objectives**

By the end of this notebook, you should be able to:

* Simulate soil moisture changes over time.
* Apply Euler's Method.
* Apply Runge-Kutta (RK4).
Perform Monte Carlo simulations.
* Formulate optimization problems.
* Determine optimal irrigation allocations.
* Interpret simulation results.

**3. Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

**4.Load Dataset**

In [ ]:
weather = pd.read_csv("weather_daily_cleaned.csv")

soil = pd.read_csv("soil_sensor_data_cleaned.csv")

crop = pd.read_csv("crop_zone_parameters_cleaned.csv")

**5. Soil Moisture Simulation
Background**

Soil moisture changes due to:

* Rainfall
* Irrigation
* Evapotranspiration
* Drainage

A simplified differential equation is:

dt/
dM
	​
=R+I−ET−D

Where:

* M = Soil Moisture
* R = Rainfall
* I = Irrigation
* ET = Evapotranspiration
* D = Drainage

**6. Euler Method**

**Theory**

Euler's Method approximates solutions of differential equations using:

**Implementation**

In [ ]:
def moisture_change(t, m):

    rainfall = 5

    irrigation = 4

    et = 3

    drainage = 1

    return rainfall + irrigation - et - drainage

**Euler Solver**

In [ ]:
def euler_method(f, y0, t0, tf, h):

    t_values = np.arange(t0, tf+h, h)

    y_values = [y0]

    y = y0

    for t in t_values[:-1]:

        y = y + h*f(t,y)

        y_values.append(y)

    return t_values, np.array(y_values)

**Execute Simulation**

In [ ]:
t_euler, moisture_euler = euler_method(
    moisture_change,
    30,
    0,
    30,
    1
)

**Visualization**

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    t_euler,
    moisture_euler,
    marker='o'
)

plt.title(
    "Soil Moisture Simulation Using Euler Method"
)

plt.xlabel("Days")

plt.ylabel("Moisture (%)")

plt.grid()

plt.show()

**Interpretation**

The simulation illustrates how soil moisture changes over time under assumed environmental conditions.

**7. Runge-Kutta Fourth Order Method (RK4)
Theory**

RK4 provides greater accuracy than Euler's Method.

The method evaluates the derivative four times during each step and combines the estimates.

**Implementation**

In [ ]:
def rk4(f, y0, t0, tf, h):

    t_values = np.arange(t0, tf+h, h)

    y_values = [y0]

    y = y0

    for t in t_values[:-1]:

        k1 = f(t,y)

        k2 = f(t+h/2, y+h*k1/2)

        k3 = f(t+h/2, y+h*k2/2)

        k4 = f(t+h, y+h*k3)

        y = y + (h/6)*(k1+2*k2+2*k3+k4)

        y_values.append(y)

    return t_values,np.array(y_values)

**Execute RK4**

In [ ]:
t_rk4, moisture_rk4 = rk4(
    moisture_change,
    30,
    0,
    30,
    1
)

**Compare Euler and RK4**

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    t_euler,
    moisture_euler,
    label="Euler"
)

plt.plot(
    t_rk4,
    moisture_rk4,
    label="RK4"
)

plt.legend()

plt.title(
    "Euler vs RK4"
)

plt.show()

**Interpretation**

RK4 generally produces more accurate solutions than Euler's Method, especially for nonlinear systems.

**8. Monte Carlo Simulation
Background**

Weather conditions are uncertain.

Monte Carlo simulation evaluates system performance under many random scenarios.

**Generate Random Rainfall**

In [ ]:
simulations = 1000

rainfall_samples = np.random.normal(
    loc=weather["rainfall_mm"].mean(),
    scale=weather["rainfall_mm"].std(),
    size=simulations
)

**Visualization**

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    rainfall_samples,
    bins=30
)

plt.title(
    "Monte Carlo Rainfall Simulation"
)

plt.xlabel("Rainfall (mm)")

plt.ylabel("Frequency")

plt.show()

**Interpretation**

The histogram represents possible future rainfall conditions based on historical observations.

**9. Water Allocation Optimization**

**Problem Statement**

Assume three irrigation zones require water.

Available water is limited.

The objective is to distribute water efficiently.

**Zone Data**

In [ ]:
crop[
[
    "zone_id",
    "target_moisture_pct",
    "area_m2"
]
]

**Objective Function**

We minimize deviation from target moisture.

In [ ]:
def objective(x):

    target = np.array([35,40,38])

    return np.sum((x-target)**2)

**Constraint**

Available water:

In [ ]:
available_water = 100

In [ ]:
constraints = (
{
'type':'eq',
'fun':lambda x: np.sum(x)-available_water
}
)

**Bounds**

In [ ]:
bounds = [
    (0,100),
    (0,100),
    (0,100)
]

**Solve Optimization**

In [ ]:
initial_guess = [30,30,40]

solution = minimize(
    objective,
    initial_guess,
    bounds=bounds,
    constraints=constraints
)

solution.x

**10. Optimization Results**

In [ ]:
results = pd.DataFrame({

    "Zone":[
        "Zone 1",
        "Zone 2",
        "Zone 3"
    ],

    "Allocated Water":
    solution.x
})

results

**visualization**

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    results["Zone"],
    results["Allocated Water"]
)

plt.title(
    "Optimized Water Allocation"
)

plt.ylabel("Water Units")

plt.show()

**Interpretation**

The optimization procedure identifies the best distribution of available water while minimizing deviations from desired moisture targets.

**11. Scenario Analysis**

**Dry Weather Scenario**

In [ ]:
dry_rainfall = weather["rainfall_mm"] * 0.5

**Wet Weather Scenario**

In [ ]:
wet_rainfall = weather["rainfall_mm"] * 1.5

**Comparison**

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    weather["rainfall_mm"],
    label="Observed"
)

plt.plot(
    dry_rainfall,
    label="Dry Scenario"
)

plt.plot(
    wet_rainfall,
    label="Wet Scenario"
)

plt.legend()

plt.title(
    "Rainfall Scenarios"
)

plt.show()

**12. Key Findings**

**Simulation**
* Soil moisture changes over time.
* RK4 generally provides better accuracy than Euler.

**Monte Carlo Analysis**
* Rainfall variability introduces uncertainty.
* Multiple future outcomes are possible.

**Optimization**
* Water allocation can be improved using mathematical optimization.
* Efficient allocation helps maintain moisture targets.

**13. Discussion**

Simulation and optimization complement each other.

Simulation predicts future behaviour while optimization identifies the best decisions.

Together they support:

* Water conservation
* Improved crop productivity
* Better irrigation scheduling
* Resource efficiency

**14. Conclusion**

This notebook demonstrated the use of simulation and optimization techniques in irrigation management.

Key achievements include:

* Modelling soil moisture dynamics.
* Comparing Euler and RK4 methods.
* Evaluating uncertainty through Monte Carlo simulation.
* Optimizing water allocation among irrigation zones.